## Prophet

In [ ]:
# ── Colab / Local setup ──────────────────────────────────────────────────
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/DaTking4/ml-final-project-walmart-recruiting.git"], check=True)
    %cd ml-final-project-walmart-recruiting
    %pip install -q -r requirements.txt

    import os
    from google.colab import userdata
    os.environ["DAGSHUB_USER_TOKEN"] = userdata.get("DAGSHUB_TOKEN")
    os.environ["WANDB_API_KEY"]      = userdata.get("WANDB_API_KEY")
    os.environ["KAGGLE_API_TOKEN"]   = userdata.get("KAGGLE_API_TOKEN")

    %pip install -q kaggle
    import os; os.makedirs("data", exist_ok=True)
    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data/ --quiet
    !unzip -q -o data/walmart-recruiting-store-sales-forecasting.zip -d data/

print("Running in:", "Google Colab" if IN_COLAB else "Local environment")


### 1. Setup and Imports

In [1]:
import os, sys, importlib, logging, warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
# W&B's background "service" process can be left in a broken state by a
# sleep/wake cycle or a crashed/killed kernel, causing BrokenPipeError on
# wandb.init() in later runs. Disabling it makes wandb run in-process
# instead, which is slightly less efficient but avoids that whole class of bug.
os.environ.setdefault("WANDB_DISABLE_SERVICE", "true")

logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "src").exists():
        sys.path.insert(0, str(repo_root)); break
    repo_root = repo_root.parent
else:
    raise FileNotFoundError("Could not locate repo root containing 'src'.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import mlflow, mlflow.pyfunc
from mlflow.models import infer_signature
import wandb

import src.mlflow_setup as mlflow_setup
importlib.reload(mlflow_setup)
init_tracking = mlflow_setup.init_tracking

from src.data_loading import load_merged
from src.transforms import apply_shared_features
from src.validation import time_based_split
from src.metrics import wmae_from_df
from src.arima_utils import to_arima_long
from src.prophet_utils import (
    make_holidays_df, fit_prophet_model, evaluate_prophet_config, fit_prophet_models,
)
from src.pipeline.prophet_pipeline import ProphetForecastPipeline

init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()
if mlflow.active_run(): mlflow.end_run()

BLUE, PINK, PURPLE, RED, GREEN = "#7196C7", "#AE737D", "#705588", "#7E3838", "#5E9D74"
REGIME_COLORS = {"underfit": PURPLE, "balanced": BLUE, "overfit": RED}
STATUS_COLORS = {"good": GREEN, "underfit": PURPLE, "overfit": RED}

print("Setup complete.")


Accessing as dkhak22

Initialized MLflow to track repo "dkhak22/ml-final-project-store-sales-forecasting"

Repository dkhak22/ml-final-project-store-sales-forecasting initialized!

MLflow tracking URI: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow
Setup complete.


### 2. Configuration

In [2]:
init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()

EXPERIMENT_NAME = "Prophet_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

WANDB_ENTITY  = "dkhak22-free-university-of-tbilisi-"
WANDB_PROJECT = "walmart-sales-forecasting"

CONFIG = {
    "horizon":          26,
    "min_train_points": 52,
}

FREQ      = "W-FRI"
MODEL_COL = "Prophet"

CONFIG


Initialized MLflow to track repo "dkhak22/ml-final-project-store-sales-forecasting"

Repository dkhak22/ml-final-project-store-sales-forecasting initialized!

MLflow tracking URI: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow


{'horizon': 26, 'min_train_points': 52}

### 3. Load Data

In [3]:
train_df, test_df = load_merged()
print(f"train_df: {train_df.shape}")
print(f"test_df:  {test_df.shape}")
CONFIG["horizon"] = test_df["Date"].nunique()
train_df.head()


train_df: (421570, 16)
test_df:  (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


### 4. Shared Preprocessing and Feature Engineering

In [4]:
train_prepared = apply_shared_features(train_df)
print(f"Columns: {train_prepared.shape[1]}")
train_prepared.head()


Columns: 23


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Unemployment,Size,Type_A,Type_B,Type_C,Year,Month,WeekOfYear,DaysSinceLastHoliday,DaysToNextHoliday
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,5,inf,7.0
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,6,0.0,0.0
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,7,7.0,203.0
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,8,14.0,196.0
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,3,9,21.0,189.0


### 5. Feature Selection

In [5]:
PROPHET_FEATURE_DECISION = {
    "feature_set": "target_history_plus_holidays",
    "uses_exogenous_features": False,
    "used_model_columns": "ds, y, holidays",
    "reason": (
        "Prophet is a univariate Bayesian structural time series model. "
        "It models each Store-Dept series independently using its own "
        "sales history. The four Walmart competition holidays (Super Bowl, "
        "Labor Day, Thanksgiving, Christmas) are passed natively via "
        "Prophet's holidays argument, giving explicit holiday effects "
        "without needing exogenous regressor columns."
    ),
}

print("Holidays passed to Prophet:")
display(make_holidays_df())
PROPHET_FEATURE_DECISION


Holidays passed to Prophet:


,holiday,ds
0,Super_Bowl,2010-02-12
1,Super_Bowl,2011-02-11
2,Super_Bowl,2012-02-10
3,Super_Bowl,2013-02-08
4,Labor_Day,2010-09-10
5,Labor_Day,2011-09-09
6,Labor_Day,2012-09-07
7,Labor_Day,2013-09-06
8,Thanksgiving,2010-11-26
9,Thanksgiving,2011-11-25


{'feature_set': 'target_history_plus_holidays',
 'uses_exogenous_features': False,
 'used_model_columns': 'ds, y, holidays',
 'reason': "Prophet is a univariate Bayesian structural time series model. It models each Store-Dept series independently using its own sales history. The four Walmart competition holidays (Super Bowl, Labor Day, Thanksgiving, Christmas) are passed natively via Prophet's holidays argument, giving explicit holiday effects without needing exogenous regressor columns."}

### 6. Time-Series and Window Setup

In [6]:
train_part, valid_part = time_based_split(train_prepared, valid_weeks=CONFIG["horizon"])

print(f"Train part: {train_part['Date'].min().date()} -> {train_part['Date'].max().date()}")
print(f"Valid part: {valid_part['Date'].min().date()} -> {valid_part['Date'].max().date()}")

long_df    = to_arima_long(train_prepared)
train_long = to_arima_long(train_part)
valid_long = to_arima_long(valid_part)

series_lengths = long_df.groupby("unique_id").size()
train_lengths  = train_long.groupby("unique_id").size()
valid_lengths  = valid_long.groupby("unique_id").size()

PROPHET_SERIES_LENGTH = int(series_lengths.max())
prophet_ids = series_lengths[series_lengths == PROPHET_SERIES_LENGTH].index
prophet_ids = prophet_ids.intersection(train_lengths[train_lengths >= CONFIG["min_train_points"]].index)
prophet_ids = prophet_ids.intersection(valid_lengths[valid_lengths == CONFIG["horizon"]].index)

# Uncomment the next line to subsample for faster iteration on Colab:
# prophet_ids = prophet_ids[:200]

print(f"Using {len(prophet_ids):,} complete-history series")
print(f"Dropping {long_df['unique_id'].nunique() - len(prophet_ids)} short/ragged series")

holiday_lookup = train_prepared.assign(
    unique_id=train_prepared["Store"].astype(str) + "_" + train_prepared["Dept"].astype(str),
    ds=train_prepared["Date"],
)[["unique_id", "ds", "IsHoliday"]].drop_duplicates()
holiday_lookup["IsHoliday"] = holiday_lookup["IsHoliday"].fillna(False).astype(bool)

train_by_id = {
    uid: grp.sort_values("ds").set_index("ds")["y"].astype(float).asfreq(FREQ)
    for uid, grp in train_long.groupby("unique_id")
    if uid in prophet_ids
}
valid_by_id = {
    uid: grp.sort_values("ds")[["unique_id", "ds", "y"]].copy()
    for uid, grp in valid_long.groupby("unique_id")
    if uid in prophet_ids
}

def fit_gap_pct(train_wmae, val_wmae):
    if pd.isna(train_wmae) or train_wmae == 0: return float("nan")
    return ((val_wmae - train_wmae) / train_wmae) * 100

def classify_fit_status(train_wmae, val_wmae):
    gap = fit_gap_pct(train_wmae, val_wmae)
    if pd.isna(gap): return "unknown"
    return "overfit" if gap > 25 else "underfit" if gap < -10 else "good"

print("Prepared grouped Prophet inputs.")


Train part: 2010-02-05 -> 2012-01-27
Valid part: 2012-02-03 -> 2012-10-26
Using 2,660 complete-history series
Dropping 671 short/ragged series
Prepared grouped Prophet inputs.


### 7. Sanity Check

In [7]:
sanity_config = {
    "label":  "sanity_prophet",
    "regime": "sanity",
    "changepoint_prior_scale": 0.05,
    "seasonality_mode": "additive",
    "yearly_seasonality": 10,
    "weekly_seasonality": False,
    "daily_seasonality": False,
}

sample_id   = list(prophet_ids)[0]
y_train_s   = train_by_id[sample_id]
valid_grp_s = valid_by_id[sample_id]

model_s = fit_prophet_model(y_train_s, sanity_config)
future  = model_s.predict(pd.DataFrame({"ds": pd.DatetimeIndex(valid_grp_s["ds"].values)}))
fc      = future["yhat"].values

assert len(fc) == CONFIG["horizon"], f"Expected {CONFIG['horizon']} steps, got {len(fc)}"
assert np.isfinite(fc).all(), "Non-finite values in forecast"

print(f"Sanity check passed for series: {sample_id}")
print(f"First 5 forecasts: {fc[:5]}")


/Users/macbookpro/Desktop/ml-final-project-walmart-recruiting/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
22:41:40 - cmdstanpy - INFO - Chain [1] start processing
22:41:40 - cmdstanpy - INFO - Chain [1] done processing


Sanity check passed for series: 10_1
First 5 forecasts: [42284.56971505 58795.26697227 46003.37006549 38511.78830586
 30878.27100131]


### 8. Baseline Run

In [8]:
baseline_config = {
    "label":  "baseline_prophet",
    "regime": "baseline",
    "changepoint_prior_scale": 0.05,
    "seasonality_prior_scale": 10.0,
    "holidays_prior_scale":    10.0,
    "seasonality_mode":        "additive",
    "yearly_seasonality":      10,
    "weekly_seasonality":      False,
    "daily_seasonality":       False,
    "changepoint_range":       0.8,
}

with mlflow.start_run(run_name="Prophet_Baseline") as run:
    wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name="Prophet_Baseline", group="Prophet", job_type="baseline",
        tags=["Prophet", "baseline", "target_history_plus_holidays"],
        config={**CONFIG, **baseline_config, **PROPHET_FEATURE_DECISION},
        reinit=True,
    )
    try:
        cv_df, val_wmae, failures, train_wmae = evaluate_prophet_config(
            config=baseline_config,
            prophet_ids=prophet_ids,
            train_by_id=train_by_id,
            valid_by_id=valid_by_id,
            holiday_lookup=holiday_lookup,
            model_col=MODEL_COL,
            min_train_points=CONFIG["min_train_points"],
        )
        gap    = fit_gap_pct(train_wmae, val_wmae)
        status = classify_fit_status(train_wmae, val_wmae)

        mlflow.log_params({k: v for k, v in baseline_config.items() if k not in ("label", "regime")})
        mlflow.log_params({"model": "Prophet", "feature_set": PROPHET_FEATURE_DECISION["feature_set"],
                           "prophet_n_series": len(prophet_ids), "gradient_logging_applicable": False})
        mlflow.log_metrics({"train_wmae": train_wmae, "val_wmae": val_wmae,
                            "gap_pct": gap, "fallback_series": failures})
        wandb.log({"train_wmae": train_wmae, "val_wmae": val_wmae,
                   "gap_pct": gap, "status": status, "fallback_series": failures})

        print(f"Baseline train WMAE: {train_wmae:,.2f}")
        print(f"Baseline val   WMAE: {val_wmae:,.2f}")
        print(f"Gap: {gap:.1f}%  Status: {status}  Fallbacks: {failures}")
    finally:
        wandb.finish()


wandb: WARNING Disabling the wandb service is deprecated as of version 0.18.0 and will be removed in future versions. 
wandb: Currently logged in as: dkhak22 (dkhak22-free-university-of-tbilisi-). Use `wandb login --relogin` to force relogin
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Evaluating 2,660 series for baseline_prophet (n_jobs=-2)


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 9 concurrent workers.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
22:41:46 - cmdstanpy - INFO - Chain [1] start processing
22:41:46 - cmdstanpy - INFO - Chain [1] start processing
Importing plotly failed. Interactive plots will not work.
22:41:46 - cmdstanpy - INFO - Chain [1] start processing
Importing plotly failed. Interactive plots will not work.
22:41:46 - cmdstanpy - INFO - Chain [1] start processing
22:41:46 - cmdstanpy - INFO - Chain [1] done processing
22:41:46 - cmdstanpy - INFO - Chain [1] start processing
22:41:46 - cmdstanpy - INFO - Chain [1] done processing
22:41:46 - cmdstanpy - INFO - Chain [1] done processing
22:41:46 - cmdstanpy - INFO - Chain [1] done processing
Importing

Baseline train WMAE: 1,216.90
Baseline val   WMAE: 1,855.10
Gap: 52.4%  Status: overfit  Fallbacks: 0


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
2026/07/12 22:42:11 INFO mlflow.tracking._tracking_service.client: 🏃 View run Prophet_Baseline at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/8/runs/ef6df4aeff734ef7886ee95d75579cbd.
2026/07/12 22:42:11 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/8.


### 9. Hyperparameters

In [9]:
param_grid = [
    # ── Underfit: rigid trend, low seasonality flexibility ──────────────────
    {"changepoint_prior_scale": 0.001, "seasonality_prior_scale": 0.1,  "holidays_prior_scale": 1.0,  "seasonality_mode": "additive",       "yearly_seasonality": 5,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_1", "regime": "underfit"},
    {"changepoint_prior_scale": 0.001, "seasonality_prior_scale": 0.01, "holidays_prior_scale": 1.0,  "seasonality_mode": "additive",       "yearly_seasonality": 5,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_2", "regime": "underfit"},
    {"changepoint_prior_scale": 0.001, "seasonality_prior_scale": 0.1,  "holidays_prior_scale": 0.1,  "seasonality_mode": "additive",       "yearly_seasonality": 3,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_3", "regime": "underfit"},
    {"changepoint_prior_scale": 0.005, "seasonality_prior_scale": 0.1,  "holidays_prior_scale": 1.0,  "seasonality_mode": "additive",       "yearly_seasonality": 5,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_4", "regime": "underfit"},
    {"changepoint_prior_scale": 0.001, "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 1.0,  "seasonality_mode": "multiplicative",  "yearly_seasonality": 3,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_5", "regime": "underfit"},
    {"changepoint_prior_scale": 0.005, "seasonality_prior_scale": 0.01, "holidays_prior_scale": 0.1,  "seasonality_mode": "additive",       "yearly_seasonality": 5,  "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "underfit_6", "regime": "underfit"},
    # ── Balanced ─────────────────────────────────────────────────────────────
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_1",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_2",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonarity": False, "changepoint_range": 0.8, "label": "balanced_3",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_4",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_5",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_6",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_7",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_8",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_9",  "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_10", "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 1.0,  "seasonality_mode": "additive",       "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_11", "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 1.0,  "seasonality_mode": "multiplicative",  "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_12", "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "additive",       "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_13", "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.8, "label": "balanced_14", "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 5.0,  "holidays_prior_scale": 5.0,  "seasonality_mode": "additive",       "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_15", "regime": "balanced"},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 5.0,  "holidays_prior_scale": 5.0,  "seasonality_mode": "multiplicative",  "yearly_seasonality": 15, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_16", "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 5.0,  "holidays_prior_scale": 5.0,  "seasonality_mode": "additive",       "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_17", "regime": "balanced"},
    {"changepoint_prior_scale": 0.1,   "seasonality_prior_scale": 5.0,  "holidays_prior_scale": 5.0,  "seasonality_mode": "multiplicative",  "yearly_seasonality": 10, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.9, "label": "balanced_18", "regime": "balanced"},
    # ── Overfit: very flexible trend and seasonality ─────────────────────────
    {"changepoint_prior_scale": 0.5,   "seasonality_prior_scale": 100.0,"holidays_prior_scale": 100.0,"seasonality_mode": "additive",       "yearly_seasonality": 25, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_1",  "regime": "overfit"},
    {"changepoint_prior_scale": 0.5,   "seasonality_prior_scale": 100.0,"holidays_prior_scale": 100.0,"seasonality_mode": "multiplicative",  "yearly_seasonality": 25, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_2",  "regime": "overfit"},
    {"changepoint_prior_scale": 0.3,   "seasonality_prior_scale": 50.0, "holidays_prior_scale": 50.0, "seasonality_mode": "additive",       "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_3",  "regime": "overfit"},
    {"changepoint_prior_scale": 0.3,   "seasonality_prior_scale": 50.0, "holidays_prior_scale": 50.0, "seasonality_mode": "multiplicative",  "yearly_seasonality": 20, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_4",  "regime": "overfit"},
    {"changepoint_prior_scale": 0.5,   "seasonality_prior_scale": 100.0,"holidays_prior_scale": 100.0,"seasonality_mode": "additive",       "yearly_seasonality": 30, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_5",  "regime": "overfit"},
    {"changepoint_prior_scale": 0.5,   "seasonality_prior_scale": 100.0,"holidays_prior_scale": 100.0,"seasonality_mode": "multiplicative",  "yearly_seasonality": 30, "weekly_seasonality": False, "daily_seasonality": False, "changepoint_range": 0.95,"label": "overfit_6",  "regime": "overfit"},
]

print(f"Total configs: {len(param_grid)}")
regime_counts = {r: sum(1 for c in param_grid if c['regime'] == r) for r in ['underfit', 'balanced', 'overfit']}
print("By regime:", regime_counts)


Total configs: 30
By regime: {'underfit': 6, 'balanced': 18, 'overfit': 6}


### 10. Prophet Experiments

In [10]:
results     = {}
cv_by_label = {}

best_val_wmae = float("inf")
best_run_id   = None
best_label    = None

with mlflow.start_run(run_name="Prophet_HyperparamSweep") as parent_run:
    mlflow.log_param("n_configs",        len(param_grid))
    mlflow.log_param("model",            "Prophet")
    mlflow.log_param("feature_set",      PROPHET_FEATURE_DECISION["feature_set"])
    mlflow.log_param("prophet_n_series", len(prophet_ids))
    mlflow.log_param("gradient_logging_applicable", False)

    for config in param_grid:
        label  = config["label"]
        regime = config["regime"]

        with mlflow.start_run(run_name=f"Prophet_{label}", nested=True) as nested_run:
            wandb.init(
                entity=WANDB_ENTITY, project=WANDB_PROJECT,
                name=f"Prophet_{label}", group="Prophet", job_type="hyperparam_sweep",
                tags=["Prophet", regime],
                config={**CONFIG, **config},
                reinit=True,
            )
            try:
                cv_df, val_wmae, failures, train_wmae = evaluate_prophet_config(
                    config=config,
                    prophet_ids=prophet_ids,
                    train_by_id=train_by_id,
                    valid_by_id=valid_by_id,
                    holiday_lookup=holiday_lookup,
                    model_col=MODEL_COL,
                    min_train_points=CONFIG["min_train_points"],
                )
                gap    = fit_gap_pct(train_wmae, val_wmae)
                status = classify_fit_status(train_wmae, val_wmae)

                mlflow.log_params({k: v for k, v in config.items() if k not in ("label", "regime")})
                mlflow.log_params({"label": label, "regime": regime,
                                   "feature_set": PROPHET_FEATURE_DECISION["feature_set"]})
                mlflow.log_metrics({"train_wmae": train_wmae, "val_wmae": val_wmae,
                                    "gap_pct": gap, "fallback_series": failures})
                wandb.log({"train_wmae": train_wmae, "val_wmae": val_wmae,
                           "gap_pct": gap, "status": status})

                cv_by_label[label] = cv_df
                results[label] = {
                    "label": label, "regime": regime, "status": status,
                    "feature_set": PROPHET_FEATURE_DECISION["feature_set"],
                    **{k: v for k, v in config.items() if k not in ("label", "regime")},
                    "train_wmae": train_wmae, "val_wmae": val_wmae,
                    "gap_pct": gap, "fallback_series": failures,
                    "run_id": nested_run.info.run_id,
                }

                if val_wmae < best_val_wmae:
                    best_val_wmae = val_wmae
                    best_run_id   = nested_run.info.run_id
                    best_label    = label

                print(f"{label:<16} train={train_wmae:,.0f}  val={val_wmae:,.0f}  gap={gap:+.1f}%  {status}")

            except Exception as exc:
                print(f"{label} FAILED: {exc}")
                mlflow.log_param("error", str(exc))
            finally:
                wandb.finish()


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
[Parallel(n_jobs=-2)]: Using backend LokyBackend with 9 concurrent workers.
22:43:05 - cmdstanpy - INFO - Chain [1] start processing
22:43:05 - cmdstanpy - INFO - Chain [1] done processing
22:43:05 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
22:43:05 - cmdstanpy - INFO - Chain [1] start processing
22:43:05 - cmdstanpy - INFO - Chain [1] start processing
22:43:05 - cmdstanpy - INFO - Chain [1] start processing
22:43:05 - cmdstanpy - INFO - Chain [1] start processing
22:43:05 - cmdstanpy - INFO - Chain [1] done processing
2

Evaluating 2,660 series for underfit_1 (n_jobs=-2)


22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
22:43:06 - cmdstanpy - INFO - Chain [1] done processing
22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] start processing
22:43:06 - cmdstanpy - INFO - Chain [1] 

underfit_1       train=1,867  val=2,792  gap=+49.6%  overfit


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
2026/07/12 22:44:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run Prophet_underfit_1 at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/8/runs/41d58c5d76cc40dd9e698af67c8953a5.
2026/07/12 22:44:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/8.
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't imp

Evaluating 2,660 series for underfit_2 (n_jobs=-2)


22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
22:44:33 - cmdstanpy - INFO - Chain [1] start processing
22:44:33 - cmdstanpy - INFO - Chain [1] done processing
22:44:33 - cmdstanpy - ERROR - Chain [1]

KeyboardInterrupt: 

### 11. Results

In [ ]:
results_df = pd.DataFrame(results.values()).sort_values("val_wmae").reset_index(drop=True)

display_cols = [
    "label", "regime", "status",
    "changepoint_prior_scale", "seasonality_prior_scale", "holidays_prior_scale",
    "seasonality_mode", "yearly_seasonality", "changepoint_range",
    "feature_set", "train_wmae", "val_wmae", "gap_pct", "fallback_series",
]
display(results_df[display_cols])

os.makedirs("reports", exist_ok=True)
results_path = "reports/prophet_results.csv"
results_df.to_csv(results_path, index=False)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(results_path)
    mlflow.log_metric("best_val_wmae", best_val_wmae)

print(f"\nBest: {best_label}  val_wmae={best_val_wmae:,.2f}")


### 12. Plots

In [ ]:
os.makedirs("Plots", exist_ok=True)

top_runs = results_df.sort_values("val_wmae").copy()
colors = [GREEN if l == best_label else REGIME_COLORS.get(r, BLUE)
          for l, r in zip(top_runs["label"], top_runs["regime"])]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_runs["label"] + " [" + top_runs["seasonality_mode"].str[:3] + "]",
             top_runs["val_wmae"], color=colors)
axes[0].invert_yaxis()
axes[0].set_xlabel("Val WMAE")
axes[0].set_title("Prophet — Val WMAE by Config")
axes[0].grid(axis="x", alpha=0.3)

for status, sdf in results_df.groupby("status"):
    axes[1].scatter(sdf["train_wmae"], sdf["val_wmae"],
                    c=STATUS_COLORS.get(status, BLUE), alpha=0.75, label=status)
lo = float(results_df[["train_wmae","val_wmae"]].min().min())
hi = float(results_df[["train_wmae","val_wmae"]].max().max())
axes[1].plot([lo, hi], [lo, hi], "k--", alpha=0.4)
axes[1].set_xlabel("Train WMAE"); axes[1].set_ylabel("Val WMAE")
axes[1].set_title("Prophet — Bias-Variance Tradeoff")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "Plots/prophet_sweep.png"
plt.savefig(plot_path, dpi=200); plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(plot_path)


### 13. Error Analysis

In [ ]:
best_cv_df = cv_by_label[best_label].copy()
best_cv_df["abs_error"] = (best_cv_df["y"] - best_cv_df[MODEL_COL]).abs()
best_cv_df[["Store", "Dept"]] = best_cv_df["unique_id"].str.split("_", n=1, expand=True)

worst_store_dept = (
    best_cv_df.groupby(["Store", "Dept"])["abs_error"]
    .mean().sort_values(ascending=False).head(15)
)
display(worst_store_dept)

holiday_error = best_cv_df.groupby("IsHoliday")["abs_error"].mean()
display(holiday_error)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metric("holiday_week_mae",    float(holiday_error.get(True,  np.nan)))
    mlflow.log_metric("nonholiday_week_mae", float(holiday_error.get(False, np.nan)))


### 14. Error Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
worst_store_dept.sort_values().plot(kind="barh", ax=ax, color=RED)
ax.set_xlabel("Mean Absolute Error")
ax.set_title("Prophet — Worst Store/Dept Validation Errors")
ax.grid(True, alpha=0.3)
plt.tight_layout()
error_plot_path = "Plots/prophet_worst_store_dept.png"
plt.savefig(error_plot_path, dpi=200); plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(error_plot_path)


### 15. Best Model

In [ ]:
print("Best label:",           best_label)
print("Best run id:",          best_run_id)
print("Best validation WMAE:", best_val_wmae)

assert best_label  is not None
assert best_run_id is not None

best_config = next(c for c in param_grid if c["label"] == best_label)
print("Best config:"); display(best_config)

fallback_by_id  = (to_arima_long(train_prepared).sort_values("ds")
                   .groupby("unique_id")["y"].last().astype(float).to_dict())
global_fallback = float(to_arima_long(train_prepared)["y"].median())

full_long_df = to_arima_long(train_prepared)

with mlflow.start_run(run_name="Prophet_FinalModel") as final_run:
    wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name="Prophet_FinalModel", group="Prophet", job_type="final_training",
        tags=["Prophet", "final"],
        config={**CONFIG, **best_config, "best_val_wmae": best_val_wmae},
        reinit=True,
    )
    try:
        final_models, final_failures = fit_prophet_models(
            full_long_df=full_long_df, ids=prophet_ids, config=best_config,
        )
        print(f"Fit {len(final_models):,} final Prophet models ({len(final_failures)} failures)")

        os.makedirs("artifacts", exist_ok=True)
        prophet_model_path = "artifacts/prophet_models.joblib"
        joblib.dump({
            "models":          final_models,
            "fallback_by_id":  fallback_by_id,
            "global_fallback": global_fallback,
        }, prophet_model_path)

        pipeline = ProphetForecastPipeline(
            fallback_by_id=fallback_by_id, global_fallback=global_fallback,
        )
        pipeline.models = final_models

        sample_input  = test_df.head(5)
        sample_output = pipeline.predict(None, sample_input)
        signature     = infer_signature(sample_input, sample_output)

        mlflow.log_params({k: v for k, v in best_config.items() if k not in ("label", "regime")})
        mlflow.log_params({"best_label": best_label, "feature_set": PROPHET_FEATURE_DECISION["feature_set"],
                           "final_failures": len(final_failures)})
        mlflow.log_metric("best_val_wmae",   best_val_wmae)
        mlflow.log_metric("n_final_models",  len(final_models))

        model_uri = mlflow.pyfunc.log_model(
            artifact_path="prophet_model",
            python_model=pipeline,
            artifacts={"prophet_model_path": prophet_model_path},
            signature=signature,
            registered_model_name="Prophet_WalmartForecast",
        ).model_uri

        wandb.log({"best_val_wmae": best_val_wmae, "n_final_models": len(final_models)})
        print(f"Model registered. URI: {model_uri}")
    finally:
        wandb.finish()


### 16. Test Loading

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_preds = loaded_model.predict(test_df.head(200))

display(loaded_preds.head())
print("Loaded prediction shape:", loaded_preds.shape)
assert set(loaded_preds.columns) == {"Id", "Weekly_Sales"}
assert loaded_preds["Weekly_Sales"].notna().all()
